In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ═══════════════════════════════════════════════════════════
import os, re, gc, glob, types
import numpy as np
import torch
import librosa
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, concatenate_datasets, load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# ✅ A100 speed boost
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
print(f"torch:        {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
BASE_DIR       = "/workspace/Fine-Tune-Whisper-small-On-Darija-using-A100-GPU"
DODA_PATH      = f"{BASE_DIR}/data/DODa/data/*.parquet"
PROCESSED_PATH = f"{BASE_DIR}/data/processed/full_dataset"
CKPT_DIR       = f"{BASE_DIR}/models/checkpoints"
OUTPUT_DIR     = f"{BASE_DIR}/models/whisper-darija-final"

os.makedirs(f"{BASE_DIR}/data/processed", exist_ok=True)
os.makedirs(CKPT_DIR,                     exist_ok=True)
os.makedirs(OUTPUT_DIR,                   exist_ok=True)
print("✅ Paths ready")

In [ ]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-medium",
    language="arabic",
    task="transcribe"
)
print("✅ Processor loaded")

In [ ]:
def normalize_ar(text):
    if not text:
        return ""
    text = text.replace("أ", "ا")
    text = text.replace("إ", "ا")
    text = text.replace("آ", "ا")
    for diacritic in ["ّ","َ","ُ","ِ","ْ","ً","ٌ","ٍ"]:
        text = text.replace(diacritic, "")
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
def preprocess(example):
    audio = example["audio"]

    if hasattr(audio, "get_all_samples"):
        samples      = audio.get_all_samples()
        audio_array  = samples.data.numpy().squeeze()
        sampling_rate = samples.sample_rate
    else:
        audio_array  = np.array(audio["array"], dtype=np.float32)
        sampling_rate = audio["sampling_rate"]

    if sampling_rate != 16000:
        audio_array = librosa.resample(
            audio_array.astype(np.float32),
            orig_sr=sampling_rate,
            target_sr=16000
        )
    audio_array = audio_array.astype(np.float32)

    # ── Audio length filter ──
    duration_sec = len(audio_array) / 16000
    if duration_sec < 0.5 or duration_sec > 31.0:
        return {"input_features": None, "labels": None}

    # ── Audio → mel spectrogram ──
    input_features = processor.feature_extractor(
        audio_array, sampling_rate=16000
    ).input_features[0]

    # ── Text normalization + tokenization ──
    text = normalize_ar(example["darija_Arab_new"])
    if not text:
        return {"input_features": None, "labels": None}

    labels = processor.tokenizer(
        text,
        language="arabic",
        task="transcribe"
    ).input_ids

    # ── Label length filter ──
    if len(labels) < 2 or len(labels) > 448:
        return {"input_features": None, "labels": None}

    return {"input_features": input_features, "labels": labels}


In [ ]:
if os.path.exists(PROCESSED_PATH):
    # ── Already preprocessed — just load ──
    print("📂 Found existing processed dataset — loading from disk...")
    full_dataset = load_from_disk(PROCESSED_PATH)
    print(f"✅ Loaded: {len(full_dataset):,} samples")

else:
    # ── First time — preprocess everything ──
    print("⚙️ First time — preprocessing datasets...")

    # ── Load DODa ──
    print("📥 Loading DODa...")
    parquet_files = glob.glob(DODA_PATH)
    print(f"   Found {len(parquet_files)} parquet files")
    raw1 = load_dataset("parquet", data_files={"train": parquet_files})["train"]
    raw1 = raw1.remove_columns([c for c in raw1.column_names if c not in ["audio","darija_Arab_new"]])
    print(f"✅ DODa: {len(raw1):,} samples")

    # ── Load Dataset 2 ──
    print("📥 Loading Dataset 2 (adiren7)...")
    ds2_default = load_dataset("adiren7/darija_speech_to_text", "default")["train"]
    ds2_dr      = load_dataset("adiren7/darija_speech_to_text", "dr")["train"]
    raw2        = concatenate_datasets([ds2_default, ds2_dr])
    raw2        = raw2.rename_column("text", "darija_Arab_new")
    raw2        = raw2.remove_columns([c for c in raw2.column_names if c not in ["audio","darija_Arab_new"]])
    print(f"✅ Dataset 2: {len(raw2):,} samples")

    # ── Merge ──
    print("🔗 Merging...")
    raw_dataset = concatenate_datasets([raw1, raw2])
    print(f"✅ Merged: {len(raw_dataset):,} samples")

    # ── Filter bad text ──
    print("🧹 Filtering bad text...")
    raw_dataset = raw_dataset.filter(
        lambda x: (
            x["darija_Arab_new"] is not None and
            x["darija_Arab_new"].strip() != "" and
            len(normalize_ar(x["darija_Arab_new"])) > 2
        ),
        desc="Filtering bad text"
    )
    print(f"✅ After filter: {len(raw_dataset):,} samples")

    # ── Preprocess ──
    print("🔄 Preprocessing (this takes 15-25 mins)...")
    full_dataset = raw_dataset.map(
        preprocess,
        remove_columns=raw_dataset.column_names,
        num_proc=4,                # ✅ A100 server has many CPUs
        writer_batch_size=200,
        desc="Preprocessing"
    )

    # ── Remove invalid samples ──
    print("🧹 Removing invalid samples...")
    before       = len(full_dataset)
    full_dataset = full_dataset.filter(
        lambda x: x["input_features"] is not None and x["labels"] is not None,
        desc="Removing invalid"
    )
    print(f"✅ Removed {before - len(full_dataset):,} invalid ({before:,} → {len(full_dataset):,})")

    # ── Shuffle ──
    full_dataset = full_dataset.shuffle(seed=42)
    print(f"✅ Shuffled")

    # ── Save ──
    print("💾 Saving to disk...")
    full_dataset.save_to_disk(PROCESSED_PATH)
    print(f"✅ Saved to {PROCESSED_PATH}")

# ── Train / Test split ──
dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
print(f"\n✅ Train: {len(dataset['train']):,}")
print(f"✅ Test:  {len(dataset['test']):,}")
print(f"✅ Total: {len(full_dataset):,}")

In [ ]:
torch.cuda.empty_cache()

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-medium",
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="eager",
)

model.config.use_cache = False
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.enable_input_require_grads()
print(f"✅ dtype:  {next(model.parameters()).dtype}")
print(f"✅ device: {next(model.parameters()).device}")

In [ ]:
def _patched_forward(
    self, input_ids=None, attention_mask=None, inputs_embeds=None,
    decoder_input_ids=None, decoder_attention_mask=None,
    decoder_inputs_embeds=None, labels=None, output_attentions=None,
    output_hidden_states=None, return_dict=None, task_ids=None, **kwargs,
):
    for key in list(kwargs.keys()):
        if hasattr(self, "special_peft_forward_args") and key in self.special_peft_forward_args:
            kwargs.pop(key)
    input_features = kwargs.pop("input_features", None)
    actual_input   = input_features if input_features is not None else input_ids
    return self.base_model(
        input_features=actual_input,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
        decoder_attention_mask=decoder_attention_mask,
        decoder_inputs_embeds=decoder_inputs_embeds,
        labels=labels,
        output_attentions=output_attentions,
        output_hidden_states=output_hidden_states,
        return_dict=return_dict,
        **kwargs,
    )

model.forward = types.MethodType(_patched_forward, model)

# ── Verify forward ──
sample      = dataset["train"][0]
test_feat   = torch.tensor([sample["input_features"]], dtype=torch.bfloat16).to("cuda:0")
test_labels = torch.tensor([sample["labels"]]).to("cuda:0")
with torch.no_grad():
    out = model(input_ids=test_feat, labels=test_labels)
print(f"✅ Forward test OK — loss: {out.loss.item():.4f}")
model.print_trainable_parameters()

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch          = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels         = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"] == 0, -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("✅ Data collator ready")

In [ ]:
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_str  = [normalize_ar(t) for t in pred_str]
    label_str = [normalize_ar(t) for t in label_str]
    return {"wer": round(float(metric.compute(predictions=pred_str, references=label_str)), 4)}

print("✅ Metrics ready")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,

    num_train_epochs=3,
    learning_rate=3e-5,
    warmup_steps=500,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    bf16=True,
    fp16=False,

    optim="adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=3,
    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=225,
    generation_num_beams=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    remove_unused_columns=False,
    label_names=["labels"],
    save_safetensors=False,
    report_to="none",
)
print("✅ Training args ready")

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer ready 🚀 Starting training...")
trainer.train(resume_from_checkpoint=True)

In [ ]:
model.generation_config.language = "arabic"
model.generation_config.task     = "transcribe"
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
model.generation_config.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")